# Week 8: Evaluation, data leakage, and comparing models honestly

**The week that separates people who train models from people who can be trusted with them.** It also answers the "is model B really better than model A?" question you said never stuck.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Splits, cross-validation, overfitting | 2 h |
| Day 2 | Data leakage: cause it on purpose, measure it, fix it | 2.5 h |
| Day 3 | Metrics: confusion matrix, precision/recall, ROC-AUC from scratch | 2 h |
| Day 4 | Comparing models: bootstrap CIs, paired permutation tests | 2 h |
| Check | **Cumulative check 2** (60 min, no hints) | 1 h |
| Buffer | Catch up on anything from Weeks 5–8 | 0.5 h+ |

**Read:** ISLP chapter 5 (resampling); scikit-learn's ["Common pitfalls"](https://scikit-learn.org/stable/common_pitfalls.html); Dror et al. (2018), section 3.

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can name 4 kinds of leakage, and fix each one with a `Pipeline` or a group split.
- [ ] You can report "B − A = x% (95% CI […]), p = …" and interpret it correctly.
- [ ] Cumulative check 2 is done.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(0)

---
# Day 1: Splits and cross-validation (2 h)

| Split | Used for | Rule |
|---|---|---|
| **Train** | fitting parameters | anything goes |
| **Validation** | choosing hyperparameters and models, early stopping | can be reused, but you slowly overfit to it |
| **Test** | the final, *one-time* estimate | **touch once**, at the very end |

With little data, **k-fold cross-validation** rotates the validation fold, so every example gets used for validation exactly once.

### Exercise 8.1: `kfold_indices`
Shuffle `range(n)` with the given seed, split it into `k` nearly equal folds, and return a list of `(train_idx, val_idx)` pairs. Each index must appear in exactly one validation fold.

In [ ]:
def kfold_indices(n, k, seed=0):
    raise NotImplementedError

In [ ]:
folds = kfold_indices(103, 5)
assert len(folds) == 5
all_val = np.concatenate([v for _, v in folds])
assert sorted(all_val) == list(range(103))
for tr, va in folds:
    assert len(set(tr) & set(va)) == 0 and len(tr) + len(va) == 103
print("✅ 8.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`idx = np.random.default_rng(seed).permutation(n)`, then `np.array_split(idx, k)`. For fold i, train on the concatenation of all the other folds.

</details>


### Overfitting, seen directly
Polynomial regression with increasing degree: training error keeps falling, while validation error forms a **U shape**.

In [ ]:
x = np.sort(rng.uniform(-1, 1, 40)); y = np.sin(3 * x) + 0.3 * rng.normal(size=40)
x_val = rng.uniform(-1, 1, 200); y_val = np.sin(3 * x_val) + 0.3 * rng.normal(size=200)
degrees = range(1, 16); tr_err, va_err = [], []
for d in degrees:
    coef = np.polyfit(x, y, d)
    tr_err.append(np.mean((np.polyval(coef, x) - y) ** 2)); va_err.append(np.mean((np.polyval(coef, x_val) - y_val) ** 2))
plt.plot(degrees, tr_err, "o-", label="train"); plt.plot(degrees, va_err, "o-", label="validation")
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.legend(); plt.title("Under- vs over-fitting"); plt.show()

✍️ *Where does the model underfit, and where does it overfit? Connect this to Week 7: what would an L2 penalty do to the degree-15 fit?* (double-click to edit)



---
# Day 2: Data leakage (2.5 h)

**Leakage** means information from the evaluation data reaches training, so the scores look better than reality. It's the **most common** reason that published medical AI results fail to replicate.

| Leak | Example | Fix |
|---|---|---|
| Preprocessing fitted on all the data | scaler, feature selection or PCA fitted before splitting | put every fitted step in a `Pipeline` |
| Group/patient leakage | images of the same patient in both train and test | split by **patient** (`GroupKFold`) |
| Tuning on the test set | picking the best of 30 configs by test score | tune on validation or with CV; test once |
| Temporal leakage | training on 2024 data to "predict" 2023 | split by time |
| Target leakage | a feature only known *after* the outcome, e.g. "treatment given" | understand where each feature comes from |

### Exercise 8.2: Make a leak, then fix it
The data below is **pure noise**: 100 samples, 5,000 random features, random labels. Any honest method should score about **50%**.
1. **Leaky:** select the 20 features most correlated with the labels **on all the data**, then run 5-fold CV on those features. Store the mean accuracy in `leaky_acc`.
2. **Honest:** put `SelectKBest(f_classif, k=20)` **inside** a `Pipeline` with the classifier, so that selection is redone on each training fold. Store the result in `honest_acc`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)

leaky_acc = None
honest_acc = None

In [ ]:
print(f"leaky {leaky_acc:.2f} vs honest {honest_acc:.2f}")
assert leaky_acc > 0.75, "the leaky version should look impressive"
assert 0.3 < honest_acc < 0.7, "the honest version should be near chance"
print("✅ 8.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Leaky: `X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)`, then `cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=5).mean()`.

</details>

<details><summary>Hint 2</summary>

Honest: `pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))`, then `cross_val_score(pipe, X_noise, y_noise, cv=5)`.

</details>


✍️ *Explain in your own words why the leaky version 'learns' from pure noise.* (double-click to edit)



### Exercise 8.3: Patient-level splits
Below: 60 synthetic "patients", each with 2–8 images. Images from the same patient look alike (a shared patient effect), and the label is a property of the **patient**. Write `group_split(groups, test_frac, seed)`, which returns `(train_idx, test_idx)` such that **no patient appears in both**. Then compare a random image-level split with your patient-level split.

In [ ]:
n_patients = 60
patient_effect = rng.normal(size=(n_patients, 30))
patient_label = rng.integers(0, 2, n_patients)
rows, groups, labels = [], [], []
for p in range(n_patients):
    for _ in range(rng.integers(2, 9)):
        rows.append(patient_effect[p] + 0.3 * rng.normal(size=30)); groups.append(p); labels.append(patient_label[p])
Xg, groups, yg = np.array(rows), np.array(groups), np.array(labels)
print(Xg.shape, "images from", n_patients, "patients; the label is random per patient")

def group_split(groups, test_frac=0.25, seed=0):
    raise NotImplementedError

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
tr, te = group_split(groups, 0.25, seed=0)
assert set(groups[tr]).isdisjoint(groups[te])
assert abs(len(set(groups[te])) - 15) <= 1
clf = KNeighborsClassifier(1)
i_tr, i_te = train_test_split(np.arange(len(yg)), test_size=0.25, random_state=0)
acc_image = clf.fit(Xg[i_tr], yg[i_tr]).score(Xg[i_te], yg[i_te])
acc_patient = clf.fit(Xg[tr], yg[tr]).score(Xg[te], yg[te])
print(f"image-level split {acc_image:.2f} vs patient-level split {acc_patient:.2f}")
assert acc_image > acc_patient + 0.2
print("✅ 8.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Shuffle the *unique* patient ids, take the first `round(test_frac * n_unique)` as test patients, then `test_idx = np.where(np.isin(groups, test_patients))[0]`.

</details>


✍️ *The labels are random per patient, so the true accuracy is 50%. Why does the image-level split score so much higher? Which capstone dataset could have this problem? (Look up whether PneumoniaMNIST splits by patient in Week 15.)* (double-click to edit)



---
# Day 3: Metrics (2 h)

|  | predicted + | predicted − |
|---|---|---|
| **actual +** | TP | FN |
| **actual −** | FP | TN |

- **Precision** (PPV) = TP/(TP+FP): "when it says positive, how often is it right?"
- **Recall** (sensitivity) = TP/(TP+FN): "what share of the real positives does it catch?"
- **F1** = the harmonic mean of the two.
- **ROC-AUC**: the probability that a random positive gets a **higher score** than a random negative. It doesn't depend on any threshold. 0.5 is chance; 1.0 is perfect ranking.

### Exercise 8.4: `precision_recall_f1`
Take 0/1 arrays. Define 0/0 as 0.

In [ ]:
def precision_recall_f1(y_true, y_pred):
    raise NotImplementedError

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
yt = rng.integers(0, 2, 200); yp = rng.integers(0, 2, 200)
p_, r_, f_, _ = precision_recall_fscore_support(yt, yp, average="binary")
assert np.allclose(precision_recall_f1(yt, yp), (p_, r_, f_))
assert precision_recall_f1(np.array([0, 0]), np.array([0, 0])) == (0.0, 0.0, 0.0)
print("✅ 8.4 passed")

### Exercise 8.5: `roc_auc` from its definition
Compute the fraction of (positive, negative) pairs where the positive has the higher score, counting ties as ½. An O(n_pos × n_neg) version with broadcasting is fine. *(Bonus: an O(n log n) version using ranks, the Mann–Whitney U statistic.)*

In [ ]:
def roc_auc(y_true, scores):
    raise NotImplementedError

In [ ]:
from sklearn.metrics import roc_auc_score
for _ in range(20):
    yt = rng.integers(0, 2, 150); s = rng.normal(size=150) + yt * rng.uniform(0, 2)
    s = np.round(s, 1)                       # create some ties
    assert abs(roc_auc(yt, s) - roc_auc_score(yt, s)) < 1e-12
print("✅ 8.5 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`pos, neg = scores[y_true == 1], scores[y_true == 0]`, then `diff = pos[:, None] - neg[None, :]`. AUC = `((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size`.

</details>


---
# Day 4: Is model B really better? (2 h)

87% versus 85% on 200 test examples is a difference of **4 examples**. Is that a real effect, or luck in which examples ended up in the test set?

**Two tools you'll use for the rest of your career:**
1. **Bootstrap confidence interval:** resample the test set *with replacement* many times, recompute the metric each time, and take the 2.5th and 97.5th percentiles. That shows how much the score would move with a different test sample.
2. **Paired permutation test:** both models are scored on the **same** examples, so compare them *per example*. Under the null hypothesis ("A and B are equally good"), swapping A's and B's outcome on any example changes nothing. Swap randomly many times, and count how often the shuffled difference is at least as large as the real one. That fraction is the p-value.

**Pairing matters:** both models find the same examples hard, so a paired test is much more sensitive than comparing two separate CIs.

### Exercise 8.6: `bootstrap_ci`
Return `(low, high)`, the percentile 95% CI of `metric(y_true[idx], y_pred[idx])` over `n_boot` resamples.

In [ ]:
def bootstrap_ci(metric, y_true, y_pred, n_boot=2000, alpha=0.05, seed=0):
    raise NotImplementedError

In [ ]:
acc = lambda a, b: np.mean(a == b)
yt = rng.integers(0, 2, 200); yp = np.where(rng.random(200) < 0.85, yt, 1 - yt)
lo, hi = bootstrap_ci(acc, yt, yp)
print(f"accuracy {acc(yt, yp):.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")
se = np.sqrt(0.85 * 0.15 / 200)
assert lo < acc(yt, yp) < hi and abs((hi - lo) - 2 * 1.96 * se) < 0.03
print("✅ 8.6 passed")

### Exercise 8.7: `paired_permutation_test`
`correct_a` and `correct_b` are 0/1 arrays (1 = that model got this example right). For each permutation, swap the pair on each example with probability ½, then compute `|mean(a') − mean(b')|`. Return the two-sided p-value: `(count + 1) / (n_perm + 1)`.

In [ ]:
def paired_permutation_test(correct_a, correct_b, n_perm=10000, seed=0):
    raise NotImplementedError

In [ ]:
ca = (rng.random(200) < 0.85).astype(int)
cb_same = ca.copy(); flip = rng.choice(np.where(ca == 0)[0], 4, replace=False); cb_same[flip] = 1   # B fixes 4 more
p_small = paired_permutation_test(ca, cb_same)
cb_big = np.where(rng.random(200) < 0.6, 1, ca)                                    # B fixes many
p_big = paired_permutation_test(ca, cb_big)
print(f"small gain p={p_small:.3f}   big gain p={p_big:.4f}")
assert p_big < 0.01 and p_small > p_big
print("✅ 8.7 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`diff = correct_a - correct_b`. Swapping a and b on an example just flips the sign of `diff` there.

</details>

<details><summary>Hint 2</summary>

Vectorised: `signs = rng.choice([-1, 1], size=(n_perm, len(diff)))`, then `perm_stats = np.abs((signs * diff).mean(axis=1))`.

</details>


### Put it together: two real models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
Xb, yb = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=1)
A = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
B = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
ca, cb = (A.predict(X_te) == y_te).astype(int), (B.predict(X_te) == y_te).astype(int)
for name, c, m in [("LogReg", ca, A), ("RandomForest", cb, B)]:
    lo, hi = bootstrap_ci(acc, y_te, m.predict(X_te))
    print(f"{name:12s} acc {c.mean():.3f}  95% CI [{lo:.3f}, {hi:.3f}]")
print(f"difference {cb.mean() - ca.mean():+.3f}, paired p = {paired_permutation_test(ca, cb):.3f}")

✍️ *Write the result the way you would in a report, in two sentences, including the CI and the p-value. Then: if you had compared 20 models and reported the best, why would that p-value be misleading? (Look up 'multiple comparisons'.)* (double-click to edit)



---
# Cumulative check 2 (60 min, no hints, no notes, no AI)

**The task:** scikit-learn's bundled `load_wine()` dataset has 3 classes and 13 features. Write a function `evaluate(X, y, seed=0)` that:
1. makes a stratified 70/30 train/test split,
2. compares `make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))` with `KNeighborsClassifier(5)` **without scaling**, using 5-fold CV on the **train** set only, and picks the better one,
3. refits the winner on the full train set and returns `(winner_name, test_accuracy, (ci_low, ci_high))` using your `bootstrap_ci`.

Then, in a markdown cell, explain why the KNN without scaling probably does badly (think about Week 5).

In [ ]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier

def evaluate(X, y, seed=0):
    raise NotImplementedError

In [ ]:
Xw, yw = load_wine(return_X_y=True)
name, test_acc, (lo, hi) = evaluate(Xw, yw)
print(name, round(test_acc, 3), (round(lo, 3), round(hi, 3)))
assert name == "logreg" and test_acc > 0.9 and lo <= test_acc <= hi
print("✅ cumulative check 2 passed")

✍️ *Time taken, and why the unscaled KNN does worse: ...* (double-click to edit)



### Month 2 reflection
Update `COMPETENCY_TRACKER.md` (the Mathematics and Machine learning rows) with evidence links, and write the Month 2 summary in `LOG.md`. **Send me** the repo link for review before you start Month 3.